In [2]:
from google.colab import drive
drive.mount('/content/drive')

import sys, importlib, shutil, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

sys.path.insert(0, '/content/drive/MyDrive/RadarGestureEdgeAI')
import radar_utils as ru
importlib.reload(ru)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')

ru.LOCAL.mkdir(exist_ok=True)
for f in ['X_compact.npy', 'y_labels.npy', 'recording_metadata.csv']:
    if not (ru.LOCAL / f).exists():
        shutil.copy2(ru.PROJECT / 'processed' / f, ru.LOCAL / f)

Xc = np.load(ru.LOCAL / 'X_compact.npy')            # full array in RAM (~330 MB)
y = np.load(ru.LOCAL / 'y_labels.npy')
meta = pd.read_csv(ru.LOCAL / 'recording_metadata.csv')
print("Xc:", Xc.shape, "| y:", y.shape)

Mounted at /content/drive
Device: cuda Tesla T4
Xc: (9593, 90, 32, 3) | y: (9593,)


In [4]:
class TinyRadarCNN(nn.Module):
    """Edge-sized CNN for compact radar input (3 x 90 x 32)."""
    def __init__(self, num_classes=12):
        super().__init__()
        def block(cin, cout, pool):
            layers = [nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                      nn.BatchNorm2d(cout), nn.ReLU()]
            if pool:
                layers.append(nn.MaxPool2d(2))
            return layers
        self.features = nn.Sequential(
            *block(3, 16, pool=True),
            *block(16, 32, pool=True),
            *block(32, 64, pool=False),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

n_params = sum(p.numel() for p in TinyRadarCNN().parameters())
print(f"TinyRadarCNN parameters: {n_params:,}  (~{n_params/1024:.1f} KB at INT8)")

TinyRadarCNN parameters: 24,476  (~23.9 KB at INT8)


In [5]:
@torch.no_grad()
def predict(model, X, bs=512):
    model.eval()
    return torch.cat([model(X[s:s + bs]).argmax(1)
                      for s in range(0, len(X), bs)]).cpu().numpy()


def to_tensor(idx, mean, std):
    x = (Xc[idx] - mean) / std                      # per-channel z-score (train stats)
    x = np.transpose(x, (0, 3, 1, 2))               # (N, 90, 32, 3) -> (N, 3, 90, 32)
    return (torch.tensor(x, dtype=torch.float32, device=DEVICE),
            torch.tensor(y[idx], device=DEVICE))


def train_fold(fold, epochs=40, patience=8, batch_size=64, lr=1e-3):
    ru.set_seed(ru.SEED + fold['fold'])
    tr, va, te = fold['train_idx'], fold['val_idx'], fold['test_idx']

    mean, std = ru.channel_stats(Xc, tr)            # training subjects only
    Xtr, ytr = to_tensor(tr, mean, std)
    Xva, yva = to_tensor(va, mean, std)
    Xte, _ = to_tensor(te, mean, std)

    model = TinyRadarCNN().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()

    best_f1, best_state, bad = -1.0, None, 0
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(Xtr), device=DEVICE)
        for s in range(0, len(perm), batch_size):
            b = perm[s:s + batch_size]
            if len(b) < 2:
                continue
            opt.zero_grad()
            loss = loss_fn(model(Xtr[b]), ytr[b])
            loss.backward()
            opt.step()
        sched.step()

        val_f1 = ru.evaluate(yva.cpu().numpy(), predict(model, Xva))['macro_f1']
        if val_f1 > best_f1:
            best_f1, bad = val_f1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    return model, predict(model, Xte), best_f1, ep, mean, std

print("Training function ready.")

Training function ready.


In [ ]:
rows, preds = [], []
t_all = time.time()

for fold in ru.loso_folds(meta):
    t0 = time.time()
    model, p, val_f1, n_ep, mean, std = train_fold(fold)
    te = fold['test_idx']
    m = ru.evaluate(y[te], p)

    rows.append({'model': 'TinyCNN', 'fold': fold['fold'],
                 'test_subject': fold['test_subject'], **m})
    preds.append(pd.DataFrame({'model': 'TinyCNN', 'fold': fold['fold'], 'idx': te,
                               'y_true': y[te], 'y_pred': p}))
    torch.save({'state_dict': model.state_dict(), 'mean': mean, 'std': std},
               ru.PROJECT / 'models' / f"tinycnn_fold{fold['fold']}.pt")

    print(f"Fold {fold['fold']} (test S{fold['test_subject']}) | epochs={n_ep:2d} | "
          f"val F1={val_f1:.3f} | TEST acc={m['accuracy']:.3f} F1={m['macro_f1']:.3f} | "
          f"{time.time() - t0:.0f}s")

cnn = pd.DataFrame(rows)
ru.save_results(cnn, 'loso_tinycnn_compact')
ru.save_results(pd.concat(preds), 'loso_tinycnn_compact_predictions')
print(f"\nTotal time: {(time.time() - t_all) / 60:.1f} min")

# Compare with Random Forest on the same compact input
rf = pd.read_csv(ru.PROJECT / 'results' / 'loso_baselines_compact.csv')
rf = rf[rf.model == 'RandomForest']
both = pd.concat([rf, cnn], ignore_index=True)

print("\n===== TinyCNN vs Random Forest (compact input, LOSO, %) =====")
print((both.groupby('model')[['accuracy', 'macro_f1']].agg(['mean', 'std']) * 100).round(1))

print("\n===== Macro-F1 per test subject (%) =====")
print((both.pivot(index='test_subject', columns='model', values='macro_f1') * 100).round(1))

Fold 1 (test S1) | epochs=13 | val F1=0.865 | TEST acc=0.707 F1=0.689 | 17s
Fold 2 (test S2) | epochs=19 | val F1=0.820 | TEST acc=0.894 F1=0.888 | 10s
Fold 3 (test S3) | epochs=13 | val F1=0.797 | TEST acc=0.818 F1=0.802 | 8s
Fold 4 (test S4) | epochs=40 | val F1=0.967 | TEST acc=0.792 F1=0.764 | 19s
Fold 5 (test S5) | epochs=28 | val F1=0.804 | TEST acc=0.934 F1=0.933 | 14s
Fold 6 (test S6) | epochs=37 | val F1=0.956 | TEST acc=0.849 F1=0.847 | 18s
Fold 7 (test S7) | epochs=36 | val F1=0.976 | TEST acc=0.960 F1=0.957 | 18s
Fold 8 (test S8) | epochs=24 | val F1=0.925 | TEST acc=0.916 F1=0.914 | 13s
Saved: /content/drive/MyDrive/RadarGestureEdgeAI/results/loso_tinycnn_compact.csv
Saved: /content/drive/MyDrive/RadarGestureEdgeAI/results/loso_tinycnn_compact_predictions.csv

Total time: 1.9 min

===== TinyCNN vs Random Forest (compact input, LOSO, %) =====
             accuracy      macro_f1     
                 mean  std     mean  std
model                                   
RandomFore

In [6]:
def prepare(idx, mode):
    """Input transform applied per recording (no cross-subject statistics)."""
    x = Xc[idx].astype(np.float32)
    if mode == 'log':
        x = np.log(x + 1e-4)                                        # compress dynamic range
    elif mode == 'persample':
        x = x / (x.std(axis=(1, 2, 3), keepdims=True) + 1e-8)       # remove overall signal strength
    return x


def train_fold_v2(fold, mode='zscore', epochs=60, patience=15, min_epochs=20,
                  batch_size=64, lr=1e-3):
    ru.set_seed(ru.SEED + fold['fold'])
    tr, va, te = fold['train_idx'], fold['val_idx'], fold['test_idx']

    xtr, xva, xte = prepare(tr, mode), prepare(va, mode), prepare(te, mode)
    mean = xtr.mean(axis=(0, 1, 2))                                 # training subjects only
    std = xtr.std(axis=(0, 1, 2))

    def to_t(x, idx):
        x = np.transpose((x - mean) / std, (0, 3, 1, 2))
        return (torch.tensor(x, dtype=torch.float32, device=DEVICE),
                torch.tensor(y[idx], device=DEVICE))

    Xtr, ytr = to_t(xtr, tr)
    Xva, yva = to_t(xva, va)
    Xte, _ = to_t(xte, te)

    model = TinyRadarCNN().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()

    best_f1, best_ep, best_state, bad = -1.0, 0, None, 0
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(Xtr), device=DEVICE)
        for s in range(0, len(perm), batch_size):
            b = perm[s:s + batch_size]
            if len(b) < 2:
                continue
            opt.zero_grad()
            loss_fn(model(Xtr[b]), ytr[b]).backward()
            opt.step()
        sched.step()

        val_f1 = ru.evaluate(yva.cpu().numpy(), predict(model, Xva))['macro_f1']
        if val_f1 > best_f1:
            best_f1, best_ep, bad = val_f1, ep, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience and ep >= min_epochs:
                break

    model.load_state_dict(best_state)
    return model, predict(model, Xte), best_f1, best_ep, ep


def run_cnn_loso(mode):
    rows, preds = [], []
    t_all = time.time()
    for fold in ru.loso_folds(meta):
        model, p, vf1, best_ep, last_ep = train_fold_v2(fold, mode=mode)
        te = fold['test_idx']
        m = ru.evaluate(y[te], p)
        rows.append({'model': f'TinyCNN-{mode}', 'fold': fold['fold'],
                     'test_subject': fold['test_subject'], **m, 'best_epoch': best_ep})
        preds.append(pd.DataFrame({'model': f'TinyCNN-{mode}', 'fold': fold['fold'],
                                   'idx': te, 'y_true': y[te], 'y_pred': p}))
        print(f"[{mode:9s}] Fold {fold['fold']} (S{fold['test_subject']}) | "
              f"best ep={best_ep:2d}/{last_ep:2d} | val F1={vf1:.3f} | "
              f"TEST F1={m['macro_f1']:.3f}")
    df = pd.DataFrame(rows)
    ru.save_results(df, f'loso_cnn_{mode}')
    ru.save_results(pd.concat(preds), f'loso_cnn_{mode}_predictions')
    print(f"[{mode}] done in {(time.time() - t_all) / 60:.1f} min\n")
    return df

print("Experiment functions ready.")

Experiment functions ready.


In [ ]:
runs = {}
for mode in ['zscore', 'log', 'persample']:
    runs[mode] = run_cnn_loso(mode)

rf = pd.read_csv(ru.PROJECT / 'results' / 'loso_baselines_compact.csv')
rf = rf[rf.model == 'RandomForest']
orig = pd.read_csv(ru.PROJECT / 'results' / 'loso_tinycnn_compact.csv')
orig['model'] = 'TinyCNN-original'

allc = pd.concat([rf, orig] + list(runs.values()), ignore_index=True)
order = ['RandomForest', 'TinyCNN-original', 'TinyCNN-zscore',
         'TinyCNN-log', 'TinyCNN-persample']

print("===== CNN DIAGNOSTIC (compact input, LOSO, %) =====")
summary = (allc.groupby('model')[['accuracy', 'macro_f1']].agg(['mean', 'std']) * 100).round(1)
print(summary.loc[order])

print("\n===== Macro-F1 per test subject (%) =====")
per = (allc.pivot(index='test_subject', columns='model', values='macro_f1') * 100).round(1)
print(per[order])

ru.save_results(allc, 'table_cnn_diagnostic')

[zscore   ] Fold 1 (S1) | best ep=36/51 | val F1=0.945 | TEST F1=0.959
[zscore   ] Fold 2 (S2) | best ep=11/26 | val F1=0.816 | TEST F1=0.888
[zscore   ] Fold 3 (S3) | best ep=29/44 | val F1=0.856 | TEST F1=0.788
[zscore   ] Fold 4 (S4) | best ep=37/52 | val F1=0.978 | TEST F1=0.706
[zscore   ] Fold 5 (S5) | best ep=20/35 | val F1=0.807 | TEST F1=0.926
[zscore   ] Fold 6 (S6) | best ep=46/60 | val F1=0.965 | TEST F1=0.831
[zscore   ] Fold 7 (S7) | best ep=43/58 | val F1=0.977 | TEST F1=0.959
[zscore   ] Fold 8 (S8) | best ep= 9/24 | val F1=0.931 | TEST F1=0.920
Saved: /content/drive/MyDrive/RadarGestureEdgeAI/results/loso_cnn_zscore.csv
Saved: /content/drive/MyDrive/RadarGestureEdgeAI/results/loso_cnn_zscore_predictions.csv
[zscore] done in 3.1 min

[log      ] Fold 1 (S1) | best ep=29/44 | val F1=0.980 | TEST F1=0.976
[log      ] Fold 2 (S2) | best ep= 8/23 | val F1=0.849 | TEST F1=0.911
[log      ] Fold 3 (S3) | best ep=31/46 | val F1=0.901 | TEST F1=0.827
[log      ] Fold 4 (S4) | b

In [7]:
import torch.nn.functional as Fnn

def augment(x, max_t=0, max_r=0, noise=0.0):
    """Random time/range shift (edge-replicate padding) + Gaussian noise.
    x: (B, 3, 90, 32) on GPU."""
    B, C, T, R = x.shape
    if max_t or max_r:
        xp = Fnn.pad(x, (max_r, max_r, max_t, max_t), mode='replicate')
        ts = torch.randint(0, 2 * max_t + 1, (B,)).tolist()
        rs = torch.randint(0, 2 * max_r + 1, (B,)).tolist()
        x = torch.stack([xp[i, :, ts[i]:ts[i] + T, rs[i]:rs[i] + R] for i in range(B)])
    if noise > 0:
        x = x + noise * torch.randn_like(x)
    return x


def train_fold_v3(fold, mode='persample', aug=None, epochs=60, patience=15,
                  min_epochs=20, batch_size=64, lr=1e-3):
    aug = aug or {}
    ru.set_seed(ru.SEED + fold['fold'])
    tr, va, te = fold['train_idx'], fold['val_idx'], fold['test_idx']

    xtr, xva, xte = prepare(tr, mode), prepare(va, mode), prepare(te, mode)
    mean = xtr.mean(axis=(0, 1, 2))
    std = xtr.std(axis=(0, 1, 2))

    def to_t(x, idx):
        x = np.transpose((x - mean) / std, (0, 3, 1, 2))
        return (torch.tensor(x, dtype=torch.float32, device=DEVICE),
                torch.tensor(y[idx], device=DEVICE))

    Xtr, ytr = to_t(xtr, tr)
    Xva, yva = to_t(xva, va)
    Xte, _ = to_t(xte, te)

    model = TinyRadarCNN().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()

    best_f1, best_ep, best_state, bad = -1.0, 0, None, 0
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(Xtr), device=DEVICE)
        for s in range(0, len(perm), batch_size):
            b = perm[s:s + batch_size]
            if len(b) < 2:
                continue
            xb = augment(Xtr[b], **aug)          # augmentation: training only
            opt.zero_grad()
            loss_fn(model(xb), ytr[b]).backward()
            opt.step()
        sched.step()

        val_f1 = ru.evaluate(yva.cpu().numpy(), predict(model, Xva))['macro_f1']
        if val_f1 > best_f1:
            best_f1, best_ep, bad = val_f1, ep, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience and ep >= min_epochs:
                break

    model.load_state_dict(best_state)
    return model, predict(model, Xte), best_f1, best_ep, ep, mean, std


def run_cnn_aug(name, aug, mode='persample'):
    rows, preds = [], []
    t_all = time.time()
    label = f'TinyCNN-aug-{name}'
    for fold in ru.loso_folds(meta):
        model, p, vf1, best_ep, last_ep, mean, std = train_fold_v3(fold, mode=mode, aug=aug)
        te = fold['test_idx']
        m = ru.evaluate(y[te], p)
        rows.append({'model': label, 'fold': fold['fold'],
                     'test_subject': fold['test_subject'], **m, 'best_epoch': best_ep})
        preds.append(pd.DataFrame({'model': label, 'fold': fold['fold'], 'idx': te,
                                   'y_true': y[te], 'y_pred': p}))
        print(f"[{name:5s}] Fold {fold['fold']} (S{fold['test_subject']}) | "
              f"best ep={best_ep:2d}/{last_ep:2d} | val F1={vf1:.3f} | "
              f"TEST F1={m['macro_f1']:.3f}")
    df = pd.DataFrame(rows)
    ru.save_results(df, f'loso_cnn_aug_{name}')
    ru.save_results(pd.concat(preds), f'loso_cnn_aug_{name}_predictions')
    print(f"[{name}] done in {(time.time() - t_all) / 60:.1f} min\n")
    return df

print("Augmentation functions ready.")

Augmentation functions ready.


In [ ]:
configs = {
    'time':  dict(max_t=10),
    'range': dict(max_r=6),
    'all':   dict(max_t=10, max_r=6, noise=0.05),
}

aug_runs = {name: run_cnn_aug(name, cfg) for name, cfg in configs.items()}

rf = pd.read_csv(ru.PROJECT / 'results' / 'loso_baselines_compact.csv')
rf = rf[rf.model == 'RandomForest']
base = pd.read_csv(ru.PROJECT / 'results' / 'loso_cnn_persample.csv')
base['model'] = 'TinyCNN-aug-none'

alla = pd.concat([rf, base] + list(aug_runs.values()), ignore_index=True)
order = ['RandomForest', 'TinyCNN-aug-none', 'TinyCNN-aug-time',
         'TinyCNN-aug-range', 'TinyCNN-aug-all']

print("===== AUGMENTATION ABLATION (persample input, LOSO, %) =====")
summary = (alla.groupby('model')[['accuracy', 'macro_f1']].agg(['mean', 'std']) * 100).round(1)
summary[('macro_f1', 'worst')] = (alla.groupby('model')['macro_f1'].min() * 100).round(1)
print(summary.loc[order])

print("\n===== Macro-F1 per test subject (%) =====")
per = (alla.pivot(index='test_subject', columns='model', values='macro_f1') * 100).round(1)
print(per[order].to_string())

ru.save_results(alla, 'table_augmentation_ablation')

[time ] Fold 1 (S1) | best ep=19/34 | val F1=0.948 | TEST F1=0.944
[time ] Fold 2 (S2) | best ep= 9/24 | val F1=0.875 | TEST F1=0.887
[time ] Fold 3 (S3) | best ep=25/40 | val F1=0.903 | TEST F1=0.810
[time ] Fold 4 (S4) | best ep=41/56 | val F1=0.963 | TEST F1=0.846
[time ] Fold 5 (S5) | best ep=28/43 | val F1=0.897 | TEST F1=0.927
[time ] Fold 6 (S6) | best ep=24/39 | val F1=0.972 | TEST F1=0.889
[time ] Fold 7 (S7) | best ep=38/53 | val F1=0.980 | TEST F1=0.954
[time ] Fold 8 (S8) | best ep=48/60 | val F1=0.967 | TEST F1=0.942
Saved: /content/drive/MyDrive/RadarGestureEdgeAI/results/loso_cnn_aug_time.csv
Saved: /content/drive/MyDrive/RadarGestureEdgeAI/results/loso_cnn_aug_time_predictions.csv
[time] done in 3.1 min

[range] Fold 1 (S1) | best ep=21/36 | val F1=0.919 | TEST F1=0.928
[range] Fold 2 (S2) | best ep=22/37 | val F1=0.864 | TEST F1=0.902
[range] Fold 3 (S3) | best ep=37/52 | val F1=0.898 | TEST F1=0.795
[range] Fold 4 (S4) | best ep=51/60 | val F1=0.964 | TEST F1=0.882
[r

In [8]:
from scipy.stats import wilcoxon

configs = {
    'none': ({}, 'loso_cnn_persample.csv'),
    'all':  (dict(max_t=10, max_r=6, noise=0.05), 'loso_cnn_aug_all.csv'),
}
extra_seeds = [1042, 2042]

parts = []
for name, (aug, seed42_file) in configs.items():
    d = pd.read_csv(ru.PROJECT / 'results' / seed42_file)      # seed 42: already done
    d['config'], d['seed'] = name, 42
    parts.append(d)
    for sd in extra_seeds:
        ru.SEED = sd                                            # temporarily change seed
        try:
            d = run_cnn_aug(f'{name}_s{sd}', aug)
        finally:
            ru.SEED = 42                                        # always restore
        d['config'], d['seed'] = name, sd
        parts.append(d)

ms = pd.concat(parts, ignore_index=True)
ru.save_results(ms, 'table_multiseed_none_vs_all')

# 1) Mean F1 for each seed
per_seed = ms.groupby(['config', 'seed'])['macro_f1'].mean() * 100
print("===== Mean LOSO macro-F1 per seed (%) =====")
print(per_seed.unstack().round(1))

print("\n===== Across 3 seeds: mean ± std of LOSO macro-F1 (%) =====")
print(per_seed.groupby('config').agg(['mean', 'std']).round(2))

# 2) Per subject, averaged over seeds
subj = (ms.groupby(['config', 'test_subject'])['macro_f1'].mean() * 100).unstack(0)
subj['diff (all - none)'] = subj['all'] - subj['none']
print("\n===== Per-subject macro-F1, averaged over 3 seeds (%) =====")
print(subj.round(1).to_string())
print("\nWorst subject:", {c: round(subj[c].min(), 1) for c in ['none', 'all']})

# 3) Paired statistical test (8 subjects)
stat, p = wilcoxon(subj['all'], subj['none'])
n_better = int((subj['diff (all - none)'] > 0).sum())
print(f"\nWilcoxon signed-rank test: p = {p:.4f}")
print(f"'all' better on {n_better}/8 subjects | mean gain = {subj['diff (all - none)'].mean():+.2f} points")

[none_s1042] Fold 1 (S1) | best ep=36/51 | val F1=0.930 | TEST F1=0.953
[none_s1042] Fold 2 (S2) | best ep=12/27 | val F1=0.828 | TEST F1=0.929
[none_s1042] Fold 3 (S3) | best ep=22/37 | val F1=0.911 | TEST F1=0.807
[none_s1042] Fold 4 (S4) | best ep=20/35 | val F1=0.958 | TEST F1=0.851
[none_s1042] Fold 5 (S5) | best ep= 5/20 | val F1=0.858 | TEST F1=0.847
[none_s1042] Fold 6 (S6) | best ep=15/30 | val F1=0.965 | TEST F1=0.876
[none_s1042] Fold 7 (S7) | best ep=25/40 | val F1=0.965 | TEST F1=0.961
[none_s1042] Fold 8 (S8) | best ep=49/60 | val F1=0.975 | TEST F1=0.944
Saved: /content/drive/MyDrive/RadarGestureEdgeAI/results/loso_cnn_aug_none_s1042.csv
Saved: /content/drive/MyDrive/RadarGestureEdgeAI/results/loso_cnn_aug_none_s1042_predictions.csv
[none_s1042] done in 2.9 min

[none_s2042] Fold 1 (S1) | best ep=23/38 | val F1=0.948 | TEST F1=0.938
[none_s2042] Fold 2 (S2) | best ep=30/45 | val F1=0.842 | TEST F1=0.955
[none_s2042] Fold 3 (S3) | best ep=39/54 | val F1=0.903 | TEST F1=0.